# Data Cleaning: Medicare Oncology Drug Spending

This notebook prepares two public CMS files for analysis in SQL (DuckDB).

| Raw file (data/raw/) | Source |
|---|---|
| `partb_spending_by_drug.csv` | Medicare Part B Spending by Drug (2018-2022) |
| `partd_spending_by_drug.csv` | Medicare Part D Spending by Drug (2018-2022) |

**Steps:** load → check → keep oncology drugs only → reshape → save clean CSVs to `data/clean/`.

In [1]:
import pandas as pd

RAW = "../data/raw/"
CLEAN = "../data/clean/"
pd.set_option("display.max_columns", 30)

## 1. Load the oncology reference table
The list of ~20 brands in scope, with class, tumour area and IRA status. Each brand has a generic-name keyword used to find it in the CMS files.

In [2]:
drugs = pd.read_csv("../data/reference/oncology_drugs.csv")
print(drugs.shape)
drugs.head()

(21, 10)


,brand,generic_keyword,generic_name,benefit,drug_class,tumour_area,manufacturer,molecule_type,fda_approval_year,ira_price_year
0,Keytruda,pembrolizumab,pembrolizumab,Part B,PD-1/PD-L1 inhibitor,Multi-tumour (immunotherapy),Merck,Biologic,2014,NaN
1,Opdivo,nivolumab,nivolumab,Part B,PD-1/PD-L1 inhibitor,Multi-tumour (immunotherapy),Bristol Myers Squibb,Biologic,2014,NaN
2,Yervoy,ipilimumab,ipilimumab,Part B,CTLA-4 inhibitor,Multi-tumour (immunotherapy),Bristol Myers Squibb,Biologic,2011,NaN
3,Tecentriq,atezolizumab,atezolizumab,Part B,PD-1/PD-L1 inhibitor,Lung,Genentech (Roche),Biologic,2016,NaN
4,Imfinzi,durvalumab,durvalumab,Part B,PD-1/PD-L1 inhibitor,Lung,AstraZeneca,Biologic,2017,NaN


In [3]:
# keyword -> brand lookup, separately for each benefit, e.g. 'Part B': {'pembrolizumab': 'Keytruda', ...}
# (some infused drugs also appear in small amounts in the Part D file, so each brand
#  is only matched in its own benefit)
keyword_to_brand = {benefit: dict(zip(group["generic_keyword"], group["brand"]))
                    for benefit, group in drugs.groupby("benefit")}

def match_brand(text, benefit):
    """Return the oncology brand of this benefit whose generic keyword appears in the text, else None."""
    text = str(text).lower()
    if "relatlimab" in text:   # Opdualag (nivolumab + relatlimab) is a different product from Opdivo
        return None
    for keyword, brand in keyword_to_brand[benefit].items():
        if keyword in text:
            return brand
    return None

match_brand("Injection, pembrolizumab, 1 mg", "Part B")

'Keytruda'

## 2. Part B Spending by Drug
One row per HCPCS billing code, with one column per metric per year (`Tot_Spndng_2018`, `Tot_Spndng_2019`, ...).

In [4]:
partb = pd.read_csv(RAW + "partb_spending_by_drug.csv", dtype=str)
print(partb.shape)
partb.iloc[:3, :8]

(635, 47)


,HCPCS_Cd,HCPCS_Desc,Brnd_Name,Gnrc_Name,Tot_Spndng_2018,Tot_Dsg_Unts_2018,Tot_Clms_2018,Tot_Benes_2018
0,90371,Hepatitis b immune globulin for injection into...,HyperHEP B*,Hepatitis B Immune Globulin*,137212.26,1365,301,139
1,90375,Rabies immune globulin for injection,HyperRAB*,Rabies Immune Globulin/PF*,7722116.19,27773,3108,3070
2,90376,Rabies immune globulin for injection beneath t...,Imogam Rabies-HT,Rabies Immune Globulin/PF,1874211.25,6252,704,696


In [5]:
# Basic checks
print("Duplicate rows:", partb.duplicated().sum())
print("Duplicate HCPCS codes:", partb["HCPCS_Cd"].duplicated().sum())
spend_cols = [c for c in partb.columns if c.startswith("Tot_Spndng_")]
print("Years in file:", [c[-4:] for c in spend_cols])
partb[["HCPCS_Cd", "Brnd_Name", "Gnrc_Name", spend_cols[-1]]].isnull().sum()

Duplicate rows: 0
Duplicate HCPCS codes: 0
Years in file: ['2018', '2019', '2020', '2021', '2022']


HCPCS_Cd            0
Brnd_Name           0
Gnrc_Name          14
Tot_Spndng_2022     0
dtype: int64

### Reshape wide → long
`melt` turns the year columns into rows, then `pivot_table` puts each metric back into its own column. The result has one row per HCPCS code per year.

In [6]:
year_cols = [c for c in partb.columns if c[-4:].isdigit() and c[-5] == "_"]

partb_long = partb.melt(id_vars=["HCPCS_Cd", "HCPCS_Desc", "Brnd_Name", "Gnrc_Name"],
                        value_vars=year_cols, var_name="column", value_name="value")
partb_long["data_year"] = partb_long["column"].str[-4:].astype(int)
partb_long["metric"] = partb_long["column"].str[:-5]
partb_long["value"] = pd.to_numeric(partb_long["value"], errors="coerce")

partb_long = partb_long.pivot_table(index=["HCPCS_Cd", "HCPCS_Desc", "Brnd_Name", "Gnrc_Name", "data_year"],
                                    columns="metric", values="value").reset_index()
partb_long.head()

metric,HCPCS_Cd,HCPCS_Desc,Brnd_Name,Gnrc_Name,data_year,Avg_Spndng_Per_Bene,Avg_Spndng_Per_Clm,Avg_Spndng_Per_Dsg_Unt,Outlier_Flag,Tot_Benes,Tot_Clms,Tot_Dsg_Unts,Tot_Spndng
0,90371,Hepatitis b immune globulin for injection into...,HyperHEP B*,Hepatitis B Immune Globulin*,2018,987.138561,455.854684,100.521802,0.0,139.0,301.0,1365.0,137212.26
1,90371,Hepatitis b immune globulin for injection into...,HyperHEP B*,Hepatitis B Immune Globulin*,2019,831.304348,428.974359,106.815642,0.0,161.0,312.0,1253.0,133840.00
2,90371,Hepatitis b immune globulin for injection into...,HyperHEP B*,Hepatitis B Immune Globulin*,2020,990.395422,559.202857,96.031332,0.0,83.0,147.0,856.0,82202.82
3,90371,Hepatitis b immune globulin for injection into...,HyperHEP B*,Hepatitis B Immune Globulin*,2021,901.178857,470.765075,111.453216,0.0,70.0,134.0,566.0,63082.52
4,90371,Hepatitis b immune globulin for injection into...,HyperHEP B*,Hepatitis B Immune Globulin*,2022,1111.549130,514.744228,135.746708,0.0,69.0,149.0,565.0,76696.89


### Total Part B drug spend (all drugs)
Saved before filtering, so the oncology share of all Part B drug spend can be calculated in SQL.

In [7]:
partb_totals = (partb_long.groupby("data_year")["Tot_Spndng"].sum()
                .reset_index().rename(columns={"Tot_Spndng": "all_drugs_spend"}))
partb_totals["benefit"] = "Part B"
partb_totals

,data_year,all_drugs_spend,benefit
0,2018,2.985322e+10,Part B
1,2019,3.586455e+10,Part B
2,2020,3.763535e+10,Part B
3,2021,3.981896e+10,Part B
4,2022,4.334437e+10,Part B


### Keep oncology drugs and tidy the columns
The milligrams per billing unit come from the code description (e.g. *'Injection, pembrolizumab, 1 mg'* → 1). They let SQL compare price per mg across drugs.

In [8]:
partb_long["brand"] = (partb_long["Gnrc_Name"] + " " + partb_long["HCPCS_Desc"]).apply(match_brand, benefit="Part B")
partb_onc = partb_long[partb_long["brand"].notna()].copy()

partb_onc["mg_per_unit"] = pd.to_numeric(partb_onc["HCPCS_Desc"].str.lower().str.extract(r"(\d+\.?\d*)\s*mg")[0])
partb_onc["benefit"] = "Part B"
partb_onc = partb_onc.rename(columns={
    "HCPCS_Cd": "hcpcs_code", "Brnd_Name": "source_name", "Tot_Spndng": "total_spend",
    "Tot_Clms": "total_claims", "Tot_Benes": "total_beneficiaries", "Tot_Dsg_Unts": "total_units"})

# a code with no spend in a year (e.g. before launch) is not a real row
partb_onc = partb_onc[partb_onc["total_spend"] > 0]
print("Oncology HCPCS codes found:", partb_onc["hcpcs_code"].nunique())
print("Brands found:", sorted(partb_onc["brand"].unique()))

Oncology HCPCS codes found: 10
Brands found: ['Darzalex', 'Enhertu', 'Imfinzi', 'Keytruda', 'Kyprolis', 'Opdivo', 'Padcev', 'Tecentriq', 'Yervoy']


## 3. Part D Spending by Drug
Same wide layout, but each drug appears once per manufacturer plus an **'Overall'** row. Only the Overall rows are kept, otherwise spend would be counted twice.

In [9]:
partd = pd.read_csv(RAW + "partd_spending_by_drug.csv", dtype=str)
print("All rows:", len(partd))
partd = partd[partd["Mftr_Name"] == "Overall"]
print("Overall rows:", len(partd))
print("Duplicate brand/generic rows:", partd.duplicated(subset=["Brnd_Name", "Gnrc_Name"]).sum())

All rows: 13889
Overall rows: 3575
Duplicate brand/generic rows: 0


In [10]:
year_cols = [c for c in partd.columns if c[-4:].isdigit() and c[-5] == "_"]

partd_long = partd.melt(id_vars=["Brnd_Name", "Gnrc_Name"], value_vars=year_cols,
                        var_name="column", value_name="value")
partd_long["data_year"] = partd_long["column"].str[-4:].astype(int)
partd_long["metric"] = partd_long["column"].str[:-5]
partd_long["value"] = pd.to_numeric(partd_long["value"], errors="coerce")
partd_long = partd_long.pivot_table(index=["Brnd_Name", "Gnrc_Name", "data_year"],
                                    columns="metric", values="value").reset_index()

partd_totals = (partd_long.groupby("data_year")["Tot_Spndng"].sum()
                .reset_index().rename(columns={"Tot_Spndng": "all_drugs_spend"}))
partd_totals["benefit"] = "Part D"

partd_long["brand"] = partd_long["Gnrc_Name"].apply(match_brand, benefit="Part D")
partd_onc = partd_long[partd_long["brand"].notna()].copy()
partd_onc["benefit"] = "Part D"
partd_onc = partd_onc.rename(columns={
    "Brnd_Name": "source_name", "Tot_Spndng": "total_spend", "Tot_Clms": "total_claims",
    "Tot_Benes": "total_beneficiaries", "Tot_Dsg_Unts": "total_units"})
partd_onc = partd_onc[partd_onc["total_spend"] > 0]
print("Brands found:", sorted(partd_onc["brand"].unique()))

Brands found: ['Calquence', 'Erleada', 'Ibrance', 'Imbruvica', 'Kisqali', 'Lenvima', 'Lynparza', 'Pomalyst', 'Tagrisso', 'Venclexta', 'Verzenio', 'Xtandi']


## 4. Combine and save the national files

In [11]:
cols = ["benefit", "brand", "source_name", "hcpcs_code", "data_year", "total_spend",
        "total_claims", "total_beneficiaries", "total_units", "mg_per_unit"]
national = pd.concat([partb_onc, partd_onc], ignore_index=True).reindex(columns=cols)

# brands in the reference table that were not found (worth checking)
missing = set(drugs["brand"]) - set(national["brand"])
print("Brands not found:", missing or "none")

# whole-number columns: pivot_table turned them into decimals (e.g. 1234.0)
count_cols = ["total_claims", "total_beneficiaries", "total_units"]
national[count_cols] = national[count_cols].round().astype("Int64")

# lineterminator="\n" keeps the files identical on Windows and Mac
national.to_csv(CLEAN + "national_spend.csv", index=False, lineterminator="\n")
pd.concat([partb_totals, partd_totals])[["benefit", "data_year", "all_drugs_spend"]] \
  .to_csv(CLEAN + "benefit_totals.csv", index=False, lineterminator="\n")
print(national.shape)
national.groupby(["benefit", "data_year"])["total_spend"].sum().unstack().round(-6)

Brands not found: none
(113, 10)


data_year,2018,2019,2020,2021,2022
benefit,,,,,
Part B,4.933000e+09,6.757000e+09,7.815000e+09,9.155000e+09,1.127900e+10
Part D,6.558000e+09,9.051000e+09,1.184300e+10,1.351400e+10,1.464500e+10


## Summary
| Clean file | One row per | Used for |
|---|---|---|
| `national_spend.csv` | brand × billing code × year | Market size, brand trends, price per mg, cost per patient, IRA |
| `benefit_totals.csv` | benefit × year | Oncology share of all drug spend |

Next step: load these into DuckDB with `sql/01_create_and_load.sql`.